## 

debug the nodes/workflow where both sessions are in one 1stlevel GLM

In [2]:
from nilearn import plotting
%matplotlib inline
from os.path import join as opj
import json
from nipype.interfaces.base import Bunch
from nipype.interfaces.spm import Level1Design, EstimateModel, EstimateContrast, SPMCommand, Info, model
from nipype.interfaces.matlab import MatlabCommand
from nipype.interfaces.freesurfer import FSCommand
from nipype.algorithms.modelgen import SpecifySPMModel, SpecifyModel
from nipype.interfaces.utility import Function, IdentityInterface
from nipype.interfaces.io import SelectFiles, DataSink
from nipype import Workflow, Node
from bids.layout import BIDSLayout
from glob import glob
from scipy import io, stats
from itertools import chain
import pandas as pd
import numpy as np
#import pytest as pt # not even needed
import nibabel as nb
import nipype
import os.path as op
from nipype.interfaces import spm

import socket
hostname = socket.gethostname()

if hostname == 'mr-02':
    MatlabCommand.set_default_paths('/home/ubuntu/matlab/spm12') # 
    bids_folder = '/mnt_01/ds-stressrisk' 
elif hostname == 'Econ117':
    MatlabCommand.set_default_paths('/Users/mrenke/matlab/spm12')
    MatlabCommand.set_default_matlab_cmd('/Applications/MATLAB_R2021b.app/bin/matlab')
    spm.SPMCommand.set_mlab_paths(matlab_cmd="/Applications/MATLAB_R2021b.app/bin/matlab") # error during spm.EstimateModel, solved: https://neurostars.org/t/nipype-spm-spmcommand-version-does-not-return-anything-on-mac/2871
    bids_folder = '/Volumes/mrenkeED/data/ds-stressrisk'

print(spm.SPMCommand().version)

12.7771


stty: 'standard input': Inappropriate ioctl for device


In [3]:
sub='01'
ses = 1
run = 1

with open(op.join(bids_folder, 'sub-01', 'ses-1','func', 'sub-01_ses-1_task-risk_run-1_bold.json'), # 'derivatives/fmriprep' , 
    "rt",
) as fp:
    task_info = json.load(fp)
TR = task_info["RepetitionTime"]
Nslices = len(task_info['SliceTiming']) # = 39
refSlice = 20 #Nslices / 2

In [4]:
def get_subject_info(subject):
    from glob import glob
    import numpy as np
    import pandas as pd
    from scipy import io, stats
    from nipype.interfaces.base import Bunch
    import os.path as op
    import os 

    bids_folder='/mnt_01/ds-stressrisk'     
    
    sub = subject
    subject_info = []

    functional_runs = []
    
    for ses in [1,2]:
        for run in range(1, 7):
            run_ses_i = (ses - 1)*6 + run
            
            # regressors (confounds + physio)
            confounds = pd.read_csv(op.join(bids_folder, 'derivatives/fmriprep',f'sub-{sub}', f'ses-{ses}', 'func', 
                            f'sub-{sub}_ses-{ses}_task-risk_run-{run}_desc-confounds_timeseries.tsv'), sep='\t')
            confound_names = ["trans_x","trans_y","trans_z","rot_x","rot_y","rot_z","a_comp_cor_00","a_comp_cor_02","a_comp_cor_03","a_comp_cor_04"]
            confounds = confounds.loc[:, confound_names]
            fn_physio = op.join(bids_folder, 'derivatives/physiotoolbox',f'sub-{sub}', f'ses-{ses}', 'func', 
                                f'sub-{sub}_ses-{ses}_task-task_run-{run}_desc-retroicor_output.mat') # task-taks (not risk) 
            physio = io.loadmat(fn_physio, simplify_cells=True)["physio"]["model"]
            physio = pd.DataFrame(
                data=physio["R"],
                columns=physio["R_column_names"])
            regressors = pd.concat([confounds, physio], axis=1)
            regressor_names = regressors.columns.values.tolist()

            # events + pmods
            df_events = pd.read_csv(op.join(bids_folder, f'sub-{sub}', f'ses-{ses}', 'func', f'sub-{sub}_ses-{ses}_task-risk_run-{run}_events.tsv'), sep='\t')
            df_events.set_index(['trial_nr', 'trial_type'], inplace=True) # only look at second option
            
            df_events_num1 = df_events.xs('stimulus 1',0,'trial_type')[['onset','prob1','n1']]
            df_risky_num1 = df_events_num1[df_events_num1['prob1'] == 0.55]
            df_safe_num1 = df_events_num1[df_events_num1['prob1'] == 1]   
            
            df_events_num2 = df_events.xs('stimulus 2',0,'trial_type')[['onset','prob2','n2']]
            df_risky_num2 = df_events_num2[df_events_num2['prob2'] == 0.55]
            df_safe_num2 = df_events_num2[df_events_num2['prob2'] == 1]

            pmod = [
                Bunch(name=['num1_risky'], param=[df_risky_num1['n1'].values.tolist()], poly=[1]),
                Bunch(name=['num1_safe'],param=[df_safe_num1['n1'].values.tolist()], poly=[1]),
                Bunch(name=['num2_risky'], param=[df_risky_num2['n2'].values.tolist()], poly=[1]),
                Bunch(name=['num2_safe'],param=[df_safe_num2['n2'].values.tolist()], poly=[1])]
            onsets = [
                df_risky_num1['onset'].values.tolist(), 
                df_safe_num1['onset'].values.tolist(),
                df_risky_num2['onset'].values.tolist(), 
                df_safe_num2['onset'].values.tolist()]

            durations = [(np.ones(len(df_risky_num1))* 0.6).tolist(), 
                         (np.ones(len(df_safe_num1))* 0.6).tolist(),
                         (np.ones(len(df_risky_num2))* 0.6).tolist(), 
                         (np.ones(len(df_safe_num2))* 0.6).tolist()]

            # put all together
            conditions = [f'risky_num1_ses{ses}', f'safe_num1_ses{ses}', f'risky_num2_ses{ses}', f'safe_num2_ses{ses}']
            subject_info.insert(
                run_ses_i - 1,
                Bunch(
                    conditions=conditions,
                    onsets=onsets,
                    durations=durations,
                    pmod=pmod,
                    tmod=None,
                    orth=['No']*len(conditions),
                    regressors=regressors.values.T.tolist(),
                    regressor_names=regressor_names,
                ),
            )

            nifti_file =  op.join(bids_folder,'derivatives/spm_nipype', f'sub-{sub}', f'ses-{ses}', f'ssub-{sub}_ses-{ses}_task-risk_run-{run}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii')
            
            #print(f'{nifti_file}')

            if op.isfile(nifti_file):
                functional_run = glob(nifti_file)[0]
                functional_runs.append(functional_run)
            else:
                print(f'{nifti_file} does not exist, probably smoothing did not work')

    return subject_info, functional_runs



In [5]:
# try function
sub = '10'
subject_info, functional_runs = get_subject_info(sub)
functional_runs

['/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-2_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-3_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-4_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-5_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-6_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-2/ssub-10_ses-2_task-risk_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressr

### start the workflow with runnning single nodes

In [7]:

getsubjectinfo = Node(
    Function(
        input_names=["subject"],
        output_names=["subject_info", "functional_runs"],
        function=get_subject_info,
    ),
    name="getsubjectinfo",
)

getsubjectinfo.inputs.subject = '03'
getsubjectinfo = getsubjectinfo.run()

240312-08:39:20,327 nipype.workflow INFO:
	 [Node] Setting-up "getsubjectinfo" in "/tmp/tmp35z2dt2x/getsubjectinfo".
240312-08:39:20,329 nipype.workflow INFO:
	 [Node] Executing "getsubjectinfo" <nipype.interfaces.utility.wrappers.Function>
240312-08:39:21,237 nipype.workflow INFO:
	 [Node] Finished "getsubjectinfo", elapsed time 0.906052s.


In [8]:
modelspec = Node(
    SpecifySPMModel(
        concatenate_runs=False,
        input_units="secs",
        output_units="secs", # 'scans' ??
        time_repetition=TR,
        high_pass_filter_cutoff=128,
    ),
    name="modelspec",
)

modelspec.inputs.subject_info = getsubjectinfo.outputs.subject_info
modelspec.inputs.functional_runs = getsubjectinfo.outputs.functional_runs

modelspec = modelspec.run()
print(modelspec.outputs.session_info)

240312-08:39:28,69 nipype.workflow INFO:
	 [Node] Setting-up "modelspec" in "/tmp/tmpke_0qksd/modelspec".
240312-08:39:28,318 nipype.workflow INFO:
	 [Node] Executing "modelspec" <nipype.algorithms.modelgen.SpecifySPMModel>
240312-08:39:28,325 nipype.workflow INFO:
	 [Node] Finished "modelspec", elapsed time 0.006176s.
[{'cond': [{'name': 'risky_num1_ses1', 'onset': [13.267996700000367, 28.402524099999937, 41.07020410000041, 53.20485139999983, 67.87304200000017, 83.50774330000058, 98.67610849999984, 112.81063220000031, 126.97874240000056, 141.6133344], 'duration': [0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6], 'pmod': [{'name': 'num1_risky', 'poly': 1, 'param': [64.0, 28.0, 79.0, 26.0, 18.0, 21.0, 56.0, 19.0, 28.0, 23.0]}]}, {'name': 'safe_num1_ses1', 'onset': [162.28210089999993, 175.91672329999986, 189.0846548999998, 204.2194172, 220.3544026, 233.02232900000035, 247.65692339999987, 262.3253567000001, 278.96028840000054, 291.1277227999999], 'duration': [0.6, 0.6, 0.6, 0.6, 0.6, 0

In [9]:
# Level1Design - Generates an SPM design matrix
level1design = Node(
    Level1Design(
        bases={"hrf": {"derivs": [0, 0]}},
        timing_units="secs",
        interscan_interval=TR,
        model_serial_correlations="AR(1)",
        microtime_resolution=Nslices,
        microtime_onset=refSlice,
        flags={"mthresh": 0.8, "globalnorm": 'None'},
        #mask_image="/mnt/d/multlearn-sns/SPM/mask_ICV.nii", ??
        volterra_expansion_order=1,
    ),
    name="level1design",
)

level1design.inputs.session_info = modelspec.outputs.session_info
level1design = level1design.run()
print(level1design.outputs.spm_mat_file)

240312-08:39:32,615 nipype.workflow INFO:
	 [Node] Setting-up "level1design" in "/tmp/tmpyxpl89i9/level1design".
240312-08:39:33,307 nipype.workflow INFO:
	 [Node] Executing "level1design" <nipype.interfaces.spm.model.Level1Design>
240312-08:41:35,533 nipype.workflow INFO:
	 [Node] Finished "level1design", elapsed time 122.223915s.


stty: 'standard input': Inappropriate ioctl for device


/tmp/tmpyxpl89i9/level1design/SPM.mat


In [12]:
# EstimateModel - estimate the parameters of the model
level1estimate = Node( EstimateModel(estimation_method={"Classical": 1},write_residuals=False), name="level1estimate")

level1estimate.inputs.spm_mat_file = level1design.outputs.spm_mat_file
level1estimate = level1estimate.run()

240312-08:42:16,726 nipype.workflow INFO:
	 [Node] Setting-up "level1estimate" in "/tmp/tmp3nmpxhgs/level1estimate".
240312-08:42:16,774 nipype.workflow INFO:
	 [Node] Executing "level1estimate" <nipype.interfaces.spm.model.EstimateModel>


stty: 'standard input': Inappropriate ioctl for device


240312-08:45:54,292 nipype.workflow INFO:
	 [Node] Finished "level1estimate", elapsed time 217.515546s.


In [15]:
def get_contrasts(subject_info):
    from nipype.interfaces.spm import EstimateContrast
    import os

    pmod_names = ['risky_num2_ses1xnum2_risky^1','risky_num2_ses2xnum2_risky^1', # 'group by riksy/safe for easier indexiing
                'safe_num2_ses1xnum2_safe^1','safe_num2_ses2xnum2_safe^1']
    condition_names = ['risky_num2_ses1','risky_num2_ses2','safe_num2_ses1','safe_num2_ses2']
    
    # same for both sessions
    con01 = ('num2_risky_int_bothSes', 'T', condition_names[0:2], [1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con02 = ('num2_safe_int_bothSes', 'T', condition_names[2:4], [1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]  
    con03 = ('num2_risky_pmod_bothSes', 'T', pmod_names[0:2], [1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con04 = ('num2_safe_pmod_bothSes', 'T', pmod_names[2:4], [1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]
    
    # difference between sessions
    con05 = ('num2_risky_int_sesDif', 'T', condition_names[0:2], [-1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con06 = ('num2_safe_int_sesDif', 'T', condition_names[2:4], [-1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]  
    con07 = ('num2_risky_sesDif', 'T', pmod_names[0:2], [-1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con08 = ('num2_safe_sesDif', 'T', pmod_names[2:4], [-1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]
       
    con_list = [con01, con02, con03, con04, con05,con06, con07, con08]
    
    return con_list

In [16]:
# EstimateContrast - estimates contrasts
getcontrasts = Node(Function(
        input_names=["subject_info"],
        output_names=["contrasts"],
        function=get_contrasts),
        name="getcontrasts")

getcontrasts.inputs.subject_info = getsubjectinfo.outputs.subject_info # output from very first node
getcontrasts = getcontrasts.run()

level1conest = Node(EstimateContrast(), name="level1conest")
level1conest.inputs.contrasts =  getcontrasts.outputs.contrasts
level1conest.inputs.spm_mat_file = level1estimate.outputs.spm_mat_file
level1conest.inputs.beta_images = level1estimate.outputs.beta_images
level1conest.inputs.residual_image = level1estimate.outputs.residual_image

level1conest = level1conest.run()
print(level1conest.outputs)

240312-09:27:24,178 nipype.workflow INFO:
	 [Node] Setting-up "getcontrasts" in "/tmp/tmpfcn43jrp/getcontrasts".
240312-09:27:24,474 nipype.workflow INFO:
	 [Node] Executing "getcontrasts" <nipype.interfaces.utility.wrappers.Function>
240312-09:27:24,478 nipype.workflow INFO:
	 [Node] Finished "getcontrasts", elapsed time 0.001231s.
240312-09:27:24,595 nipype.workflow INFO:
	 [Node] Setting-up "level1conest" in "/tmp/tmpyf19tz0z/level1conest".
240312-09:27:24,775 nipype.workflow INFO:
	 [Node] Executing "level1conest" <nipype.interfaces.spm.model.EstimateContrast>


stty: 'standard input': Inappropriate ioctl for device


240312-09:28:00,781 nipype.workflow INFO:
	 [Node] Finished "level1conest", elapsed time 36.002991s.

con_images = ['/tmp/tmpyf19tz0z/level1conest/con_0001.nii', '/tmp/tmpyf19tz0z/level1conest/con_0002.nii', '/tmp/tmpyf19tz0z/level1conest/con_0003.nii', '/tmp/tmpyf19tz0z/level1conest/con_0004.nii', '/tmp/tmpyf19tz0z/level1conest/con_0005.nii', '/tmp/tmpyf19tz0z/level1conest/con_0006.nii', '/tmp/tmpyf19tz0z/level1conest/con_0007.nii', '/tmp/tmpyf19tz0z/level1conest/con_0008.nii']
ess_images = <undefined>
spmF_images = <undefined>
spmT_images = ['/tmp/tmpyf19tz0z/level1conest/spmT_0001.nii', '/tmp/tmpyf19tz0z/level1conest/spmT_0002.nii', '/tmp/tmpyf19tz0z/level1conest/spmT_0003.nii', '/tmp/tmpyf19tz0z/level1conest/spmT_0004.nii', '/tmp/tmpyf19tz0z/level1conest/spmT_0005.nii', '/tmp/tmpyf19tz0z/level1conest/spmT_0006.nii', '/tmp/tmpyf19tz0z/level1conest/spmT_0007.nii', '/tmp/tmpyf19tz0z/level1conest/spmT_0008.nii']
spm_mat_file = /tmp/tmpyf19tz0z/level1conest/SPM.mat



## connect all Nodes to a workflow

In [29]:
getsubjectinfo = Node(
    Function(
        input_names=["subject"],
        output_names=["subject_info", "functional_runs"],
        function=get_subject_info,
    ),
    name="getsubjectinfo")

modelspec = Node(
    SpecifySPMModel(
        concatenate_runs=False,
        input_units="secs",
        output_units="secs", # 'scans' ??
        time_repetition=TR,
        high_pass_filter_cutoff=128,
    ),
    name="modelspec")

level1design = Node(
    Level1Design(
        bases={"hrf": {"derivs": [0, 0]}},
        timing_units="secs",
        interscan_interval=TR,
        model_serial_correlations="AR(1)",
        microtime_resolution=Nslices,
        microtime_onset=refSlice,
        flags={"mthresh": 0.8, "globalnorm": 'None'},
        #mask_image="/mnt/d/multlearn-sns/SPM/mask_ICV.nii", ??
        volterra_expansion_order=1,),
        name="level1design")

level1estimate = Node( 
    EstimateModel(estimation_method={"Classical": 1},write_residuals=False),
     name="level1estimate")

getcontrasts = Node(Function(
        input_names=["subject_info"],
        output_names=["contrasts"],
        function=get_contrasts),
        name="getcontrasts")

level1conest = Node(
    EstimateContrast(), name="level1conest")


In [30]:
from nipype.interfaces.io import DataSink
from nipype.interfaces.utility import IdentityInterface
import os 
#subject_list = layout.get_subjects()
subject_list =['02','03']

infosource = Node(
    IdentityInterface( fields=["subject_id"]),
    name="infosource",)
infosource.iterables = [ ("subject_id", subject_list)]


base_dir = "/mnt_01/stressrisk_wf/spm_nipype"
output_dir = 'model1'
working_dir = 'workingdir'

if not os.path.exists(base_dir):
    os.makedirs(base_dir)
    
datasink = Node(DataSink(base_directory=base_dir,
                         container=output_dir),
                name="datasink")
substitutions = [('_subject_id_', 'sub-')]
subjFolders = [('_sub-%s' % (sub), 'sub-%s' % (sub))
            for sub in subject_list]
substitutions.extend(subjFolders)
datasink.inputs.substitutions = substitutions

In [32]:
# Create a Nipype workflow
first_level_wf = Workflow(name='first_level_wf')

#getsubjectinfo.inputs.subject = '02'

# Connect the nodes
first_level_wf.connect([
    (infosource, getsubjectinfo, [("subject_id", "subject")]),
    (getsubjectinfo, modelspec, [('subject_info', 'subject_info'), ('functional_runs', 'functional_runs')]),
    (modelspec, level1design, [('session_info', 'session_info')]),
    (getsubjectinfo, getcontrasts, [('subject_info', 'subject_info')]),
    (level1design, level1estimate, [('spm_mat_file', 'spm_mat_file')]),  # Connect level1design to level1estimate
    (getcontrasts, level1conest, [('contrasts', 'contrasts')]),  # Connect the contrasts to EstimateContrast
    (level1estimate, level1conest, [('spm_mat_file', 'spm_mat_file'),('beta_images', 'beta_images'),('residual_image', 'residual_image')]),
    (level1conest, datasink, [('spm_mat_file', '1stLevel.@spm_mat'),
                                        ('spmT_images', '1stLevel.@T'),
                                        ('con_images', '1stLevel.@con')]), 
])

first_level_wf.config['logging'] = {'workflow_level' : 'DEBUG',
                        'filemanip_level' : 'DEBUG',
                        'interface_level' : 'DEBUG',
                        'log_to_file' : 'True',
                        'log_directory' : '/output/log_folder'}
                        
first_level_wf.config['execution'] = {'stop_on_first_rerun': 'True',
                        'local_hash_check':'True',
                        'hash_method': 'timestamp'}

first_level_wf.run()

240312-11:15:40,915 nipype.workflow INFO:
	 Workflow first_level_wf settings: ['check', 'execution', 'logging', 'monitoring']
240312-11:15:40,930 nipype.workflow INFO:
	 Running serially.
240312-11:15:40,930 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.getsubjectinfo" in "/tmp/tmp7uswjo0s/first_level_wf/_subject_id_02/getsubjectinfo".
240312-11:15:40,933 nipype.workflow INFO:
	 [Node] Executing "getsubjectinfo" <nipype.interfaces.utility.wrappers.Function>
240312-11:15:41,696 nipype.workflow INFO:
	 [Node] Finished "getsubjectinfo", elapsed time 0.761919s.
240312-11:15:41,765 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.getsubjectinfo" in "/tmp/tmpham43p2x/first_level_wf/_subject_id_03/getsubjectinfo".
240312-11:15:41,767 nipype.workflow INFO:
	 [Node] Executing "getsubjectinfo" <nipype.interfaces.utility.wrappers.Function>
240312-11:15:42,561 nipype.workflow INFO:
	 [Node] Finished "getsubjectinfo", elapsed time 0.793016s.
240312-11:15:42,632 nipype.workflow 

stty: 'standard input': Inappropriate ioctl for device


240312-11:16:32,916 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.level1design" in "/tmp/tmpksw_7elg/first_level_wf/_subject_id_03/level1design".
240312-11:16:33,736 nipype.workflow INFO:
	 [Node] Executing "level1design" <nipype.interfaces.spm.model.Level1Design>
240312-11:18:10,681 nipype.workflow INFO:
	 [Node] Finished "level1design", elapsed time 96.943052s.


stty: 'standard input': Inappropriate ioctl for device


240312-11:18:10,886 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.level1estimate" in "/tmp/tmpeg6fw2_j/first_level_wf/_subject_id_02/level1estimate".
240312-11:18:10,907 nipype.workflow INFO:
	 [Node] Executing "level1estimate" <nipype.interfaces.spm.model.EstimateModel>


...

240312-11:25:13,868 nipype.workflow INFO:
	 [Node] Finished "level1estimate", elapsed time 203.256066s.
240312-11:25:13,888 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.level1conest" in "/tmp/tmp9o44b8vg/first_level_wf/_subject_id_02/level1conest".
240312-11:25:14,30 nipype.workflow INFO:
	 [Node] Executing "level1conest" <nipype.interfaces.spm.model.EstimateContrast>


stty: 'standard input': Inappropriate ioctl for device


240312-11:25:47,898 nipype.workflow INFO:
	 [Node] Finished "level1conest", elapsed time 33.865578s.
240312-11:25:47,913 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.level1conest" in "/tmp/tmp953zajtr/first_level_wf/_subject_id_03/level1conest".
240312-11:25:48,107 nipype.workflow INFO:
	 [Node] Executing "level1conest" <nipype.interfaces.spm.model.EstimateContrast>


stty: 'standard input': Inappropriate ioctl for device


240312-11:26:19,965 nipype.workflow INFO:
	 [Node] Finished "level1conest", elapsed time 31.856885s.
240312-11:26:19,983 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.datasink" in "/tmp/tmp1rnzw2e5/first_level_wf/_subject_id_02/datasink".
240312-11:26:19,989 nipype.workflow INFO:
	 [Node] Executing "datasink" <nipype.interfaces.io.DataSink>
240312-11:26:19,990 nipype.interface INFO:
	 sub: /mnt_01/stressrisk_wf/spm_nipype/model1/1stLevel/_subject_id_02/SPM.mat -> /mnt_01/stressrisk_wf/spm_nipype/model1/1stLevel/sub-02/SPM.mat
240312-11:26:20,63 nipype.interface INFO:
	 sub: /mnt_01/stressrisk_wf/spm_nipype/model1/1stLevel/_subject_id_02/spmT_0001.nii -> /mnt_01/stressrisk_wf/spm_nipype/model1/1stLevel/sub-02/spmT_0001.nii
240312-11:26:20,66 nipype.interface INFO:
	 sub: /mnt_01/stressrisk_wf/spm_nipype/model1/1stLevel/_subject_id_02/spmT_0002.nii -> /mnt_01/stressrisk_wf/spm_nipype/model1/1stLevel/sub-02/spmT_0002.nii
240312-11:26:20,68 nipype.interface INFO:
	 sub: /mnt_01

In [33]:
crash_file = glob('*-level1conest-*.pklz')
crash_file

['crash-20240312-093343-ubuntu-level1conest-40b63e31-3447-4027-bcff-80a052f8f7b8.pklz',
 'crash-20240312-100522-ubuntu-level1conest-64fb28c8-81b3-4496-a47b-3f4585f1d2a3.pklz']

In [45]:
crash_file = glob('*1305*-getsub*.pklz')
crash_file

['crash-20240312-130538-ubuntu-getsubjectinfo.a34-af5ae6f2-80c7-44bf-ad3b-c52ed2aa88d3.pklz']

In [47]:
from nipype.utils.filemanip import loadpkl

#crash_file = 'crash-20240312-124753-ubuntu-getsubjectinfo.a39-f3dd5f1f-94ec-4fc7-a8ca-08ef54be865e.pklz'
res = loadpkl(crash_file[0])

In [43]:
res

{'node': first_level_wf.getsubjectinfo.a39,
 'traceback': ['Traceback (most recent call last):\n',
  '  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/pipeline/plugins/multiproc.py", line 67, in run_node\n    result["result"] = node.run(updatehash=updatehash)\n',
  '  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/pipeline/engine/nodes.py", line 527, in run\n    result = self._run_interface(execute=True)\n',
  '  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/pipeline/engine/nodes.py", line 645, in _run_interface\n    return self._run_command(execute)\n',
  '  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/pipeline/engine/nodes.py", line 771, in _run_command\n    raise NodeExecutionError(msg)\n',
  'nipype.pipeline.engine.nodes.NodeExecutionError: Exception raised while executing Node getsubjectinfo.\n\nTraceback:\n\tTraceback (most recent call las

In [41]:
sub = '01'
ses=1
run=1
nifti_file =  op.join(bids_folder,'derivatives/spm_nipype', f'sub-{sub}', f'ses-{ses}', f'ssub-{sub}_ses-{ses}_task-risk_run-{run}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii')
functional_run = glob(nifti_file)[0]
functional_run

'/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-01/ses-1/ssub-01_ses-1_task-risk_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii'